# Bibliotecas, Parâmetros e Funções

## Importação de bibliotecas

In [ ]:
import os
import pathlib
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import pandas as pd
import requests

## Parâmetros

In [ ]:
# Caminhos locais
DIR_DADOS_ORIGEM: str = "../data/raw/"
DIR_DADOS_DESTINO: str = "../data/raw/"
NOME_ARQUIVO_DESTINO: str = "SPDadosCriminais_Unificado.csv"

# Parâmetros de download dos arquivos
BASE_URL: str = "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados"
NOME_ARQUIVO: str = "SPDadosCriminais_{ano}.xlsx"
ANO_INICIO: int = 2022
ANO_FIM: int = datetime.now().year

# Parâmetros de multiprocessamento
MAX_WORKERS: int = 4

# Parâmetros de leitura dos arquivos
FORMATO_DATAS = "ISO-8601"


In [ ]:
# Colunas do dataset
COLUNAS_CATEGORICAS = [
    "NOME_SECCIONAL",
    "NOME_DELEGACIA",
    "CIDADE",
    "NUM_BO",
    "ANO_BO",
    "HORA_OCORRENCIA_BO",
    "DESCR_PERIODO",
    "DESCR_SUBTIPOLOCAL",
    "BAIRRO",
    "LOGRADOURO",
    "NOME_DELEGACIA_CIRCUNCRIÇÃO",
    "NOME_DEPARTAMENTO_CIRCUNCRIÇÃO",
    "NOME_SECCIONAL_CIRCUNCRIÇÃO",
    "NOME_MUNICIPIO_CIRCUNCRIÇÃO",
    "RUBRICA",
    "DESCR_CONDUTA",
    "NATUREZA_APURADA",
    "ANO_ESTATISTICA",
    "CMD",
    "BTL",
    "CIA",
    "CD_IBGE",
]

COLUNAS_DATA = [
    "DATA_COMUNICACAO_BO",
    "DATA_OCORRENCIA_BO",
]

COLUNAS_NUMERICAS = [
    "LATITUDE",
    "LONGITUDE",
]

## Funções

In [ ]:
def download_arquivo(ano: int, pasta_destino: str = DIR_DADOS_ORIGEM) -> str:
    nome_arquivo = NOME_ARQUIVO.format(ano=ano)
    url = f"{BASE_URL}/{nome_arquivo}"
    path_pasta_destino = pathlib.Path(pasta_destino)
    caminho_local = path_pasta_destino / nome_arquivo

    path_pasta_destino.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(caminho_local, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # blocos de 1 MB
                if chunk:
                    f.write(chunk)

    return f"Ano {ano} baixado com sucesso em {caminho_local}."

In [ ]:
def encontrar_arquivos_xlsx(diretorio: str | os.PathLike) -> list[pathlib.Path]:
    diretorio_path = pathlib.Path(diretorio)

    print(f"Procurando arquivos .xlsx em '{diretorio_path.resolve()}'...")
    arquivos = sorted(diretorio_path.glob("*.xlsx"))

    if not arquivos:
        msg_erro = f"No .xlsx arquivos found in '{diretorio_path}'."
        print(msg_erro)
        raise FileNotFoundError(msg_erro)

    print(f"{len(arquivos)} arquivos(s) encontrado(s) para processar.")
    return arquivos

In [ ]:
def extrair_ano_do_nome_do_arquivo(filename: str) -> int | None:
    match = re.search(r"\d{4}", filename)
    if match:
        year = int(match.group())
        return year

    print("Não foi possível identificar o ano do arquivo '%s'.", filename)
    return None

In [ ]:
def ler_abas_de_semestre(filepath: str | os.PathLike) -> tuple[list[pd.DataFrame], str]:
    """
    Lê a 2ª e a 3ª abas de um arquivo Excel (semestres), adicionando metadados de origem.
    Retorna os DataFrames lidos.
    """
    ano = extrair_ano_do_nome_do_arquivo(filepath.name)

    excel_obj = pd.ExcelFile(filepath, engine="openpyxl")
    abas = excel_obj.sheet_names

    dfs = []
    abas_de_semestre = abas[1:]

    for idx_semestre, aba in enumerate(abas_de_semestre, start=1):
        df_sheet = pd.read_excel(excel_obj, sheet_name=aba)

        df_sheet.columns = df_sheet.columns.astype(str).str.strip()

        # Metadados
        df_sheet["ano_referencia"] = ano
        df_sheet["semestre_referencia"] = idx_semestre
        df_sheet["arquivo_origem"] = filepath.name

        dfs.append(df_sheet)

    return (
        dfs,
        f"Arquivo {filepath.name} processado ({len(dfs)} abas, {sum(len(df) for df in dfs)} registros).",
    )

# Download dos arquivos

In [ ]:
anos = list(range(ANO_INICIO, ANO_FIM + 1))

print(f"Iniciando download dos arquivos Excel dos anos: {anos}")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futuros = {executor.submit(download_arquivo, ano): ano for ano in anos}

    for futuro in as_completed(futuros):
        ano = futuros[futuro]
        try:
            resultado = futuro.result()
            print(resultado)
        except requests.RequestException as erro:
            print(f"Erro ao baixar o ano {ano}: {erro}")

print("Download concluído.")

# Unificação de arquivos

In [ ]:
# arquivos = encontrar_arquivos_xlsx(DIR_DADOS_ORIGEM)

# df_unificado = pd.DataFrame()

# print("Iniciando unificação dos datasets")

# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#     futuros = {
#         executor.submit(ler_abas_de_semestre, arquivo): arquivo for arquivo in arquivos
#     }

#     for futuro in as_completed(futuros):
#         arquivo = futuros[futuro]
#         try:
#             dfs, resultado = futuro.result()
#             print(resultado)

#             for df in dfs:
#                 if isinstance(df, pd.DataFrame):
#                     df_unificado = pd.concat(
#                         [df_unificado, df],
#                         ignore_index=True,
#                         sort=False,
#                     )
#         except (
#             FileNotFoundError,
#             OSError,
#             ValueError,
#             TypeError,
#             KeyError,
#             pd.errors.EmptyDataError,
#             pd.errors.ParserError,
#         ) as erro:
#             print(f"Erro ao processar o arquivo {arquivo}: {erro}")

# print("Unificação concluída")

In [ ]:
arquivos = encontrar_arquivos_xlsx(DIR_DADOS_ORIGEM)

df_unificado = pd.DataFrame()

print("Iniciando unificação dos datasets")

for arquivo in arquivos:
    try:
        dfs, resultado = ler_abas_de_semestre(arquivo)
        print(resultado)

        for df in dfs:
            if isinstance(df, pd.DataFrame):
                df_unificado = pd.concat(
                    [df_unificado, df],
                    ignore_index=True,
                    sort=False,
                )
    except (
        FileNotFoundError,
        OSError,
        ValueError,
        TypeError,
        KeyError,
        pd.errors.EmptyDataError,
        pd.errors.ParserError,
    ) as erro:
        print(f"Erro ao processar o arquivo {arquivo}: {erro}")

In [ ]:
df_unificado.head()

# Tratamento de dados

In [ ]:
df_tratado = df_unificado.copy()

for col in COLUNAS_CATEGORICAS:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].astype("category")

for col in COLUNAS_DATA:
    if col in df_tratado.columns:
        df_tratado[col] = pd.to_datetime(
            df_tratado[col], format="mixed", yearfirst=True, errors="coerce"
        )

for col in COLUNAS_NUMERICAS:
    if col in df_tratado.columns:
        df_tratado[col] = pd.to_numeric(
            df_tratado[col], errors="coerce"
        )

# Exportação dos dados tratados

In [ ]:
arquivo_destino = pathlib.Path(DIR_DADOS_ORIGEM) / NOME_ARQUIVO_DESTINO

df_tratado.to_csv(arquivo_destino, sep=";", index=False, encoding="utf-8")